# OASIS production-tool exploration

This notebook inspects the same registered tools used by the formal pipeline. It does not provide runtime state to the CLI, FastAPI, or Agent. The default router is Haversine so running all cells does not consume ORS quota.

## 1. Environment and project root

In [7]:
import sys
from pathlib import Path
from dotenv import load_dotenv

START_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (START_DIR, *START_DIR.parents)
                     if (path / "oasis").is_dir() and (path / "config").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Cannot locate the OASIS project root from {START_DIR}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")
print("python:", sys.executable)
print("project root:", PROJECT_ROOT)

python: c:\Users\25145857r\.conda\envs\oasis-agent\python.exe
project root: D:\Research\ACM SIGSPATIAL\oasis-agent


## 2. Confirmed task and isolated Context

Set `RUN_ORS = True` only when an ORS key is available and live routing is intended. The production safeguards and place-scoped cache still apply.

In [8]:
from copy import deepcopy
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oasis.agent.context import Context
from oasis.contracts.domain import TaskSpec
from oasis.services.registry import registry
from oasis.settings import get_weights, load_cfg

PLACE_NAME = "Los Angeles County, California, USA"
N_FACILITIES = 10
THRESHOLD_MINUTES = 15
RUN_ORS = False

cfg = deepcopy(load_cfg())
cfg["accessibility"]["router"] = "ors" if RUN_ORS else "haversine"
cfg["output"]["dir"] = str(PROJECT_ROOT / "outputs_run" / "notebook")

task = TaskSpec(
    place_name=PLACE_NAME,
    n_facilities=N_FACILITIES,
    threshold_minutes=THRESHOLD_MINUTES,
    weights=get_weights("balanced"),
)
ctx = Context()
ctx.put("config", cfg)
ctx.put("task", task)
print(task.model_dump())

{'place_name': 'Los Angeles County, California, USA', 'intervention': 'cooling_center', 'analysis_goal': 'facility_siting', 'candidate_source': 'osm', 'candidate_file': None, 'n_facilities': 10, 'threshold_minutes': 15.0, 'coverage_target': None, 'coverage_group': 'all', 'weights': {'population_density': 0.4, 'elderly_ratio': 0.2, 'poverty_rate': 0.2, 'heat_exposure': 0.2}, 'candidate_amenities': ['library', 'community_centre', 'school'], 'regional_requirements': [], 'preference_question': '', 'notes': ''}


In [9]:
def run_tool(name, **kwargs):
    """Run one registered tool and surface any structured failure."""
    result = registry.get(name).fn(ctx=ctx, **kwargs)
    print(name, "ok=" + str(result.ok), result.summary)
    if not result.ok:
        raise RuntimeError(result.error.model_dump() if result.error else f"{name} failed")
    return result

## 3. Resolve the study area

The tool resolves the canonical boundary, FIPS codes, administrative level, and cache scope together. No FIPS value is hard-coded in downstream cells.

In [10]:
run_tool("resolve_study_area", place_name=task.place_name)
study = ctx.get("study")
print({key: study[key] for key in ("place_name", "place_type", "state_fips", "county_fips", "county_fips_list", "cache_scope", "area_km2")})
study["boundary"].plot(figsize=(8, 8), facecolor="none", edgecolor="#185FA5")
plt.title(study["place_name"])
plt.axis("off")
plt.show()

resolve_study_area ok=True {'state_fips': '06', 'county_fips': '037', 'county_fips_list': ['037'], 'place_type': 'county', 'admin_level': 6, 'area_km2': 12294.281007456097, 'cache_scope': 'los_angeles_county_california_usa', 'canonical_name': 'Los Angeles County, California, USA', 'year': 2022}
{'place_name': 'Los Angeles County, California, USA', 'place_type': 'county', 'state_fips': '06', 'county_fips': '037', 'county_fips_list': ['037'], 'cache_scope': 'los_angeles_county_california_usa', 'area_km2': 12294.281007456097}


C:\Users\25145857r\AppData\Local\Temp\ipykernel_12620\3104632329.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Prepare Census and Landsat demand

`prepare_demand` performs the scoped ACS/TIGER/Landsat fetch, validates the join, imputes missing tract temperatures, and stores aligned demand objects in Context.

In [11]:
run_tool("prepare_demand")
demand = ctx.get("demand")
areas = demand["areas"]
points = demand["points"]
population = demand["population"]
vulnerable_mask = demand["vulnerable_mask"]
indicators = demand["indicators"]

print("tracts:", len(areas))
print("CRS:", areas.crs)
print("heat quality:", ctx.get("quality")["heat"])
print(areas[["GEOID", "total_population", "elderly_ratio", "poverty_rate", "heat_exposure"]].head())

prepare_demand ok=True {'tracts': 2476, 'vulnerable_tracts': 619, 'heat_missing_tracts': 0, 'heat_imputation': 'none'}
tracts: 2476
CRS: EPSG:4326
heat quality: {'missing_tracts': 0, 'total_tracts': 2498, 'imputation': 'none'}
         GEOID  total_population  elderly_ratio  poverty_rate  heat_exposure
0  06037137504              1983       0.171962      0.017650      43.882860
1  06037138000              4826       0.235599      0.056776      44.588456
2  06037139200              5946       0.181971      0.082072      50.133712
3  06037143200              4235       0.072727      0.096340      46.414019
4  06037201301              3433       0.198660      0.142944      45.876045


In [12]:
assert len(areas) == len(points) == len(population)
assert areas["GEOID"].is_unique
assert areas.geometry.notna().all() and not areas.geometry.is_empty.any()
assert points.geometry.geom_type.eq("Point").all()
assert np.isfinite(population).all()
assert all(len(values) == len(areas) and np.isfinite(values).all() for values in indicators.values())
print("demand validation: ok")

demand validation: ok


### Older-adult population share

In [13]:
bins = [0, .10, .15, .20, .25, np.inf]
labels = ["Below 10%", "10% to <15%", "15% to <20%", "20% to <25%", "25% or above"]
elderly_view = areas.copy()
elderly_view["elderly_group"] = pd.cut(elderly_view["elderly_ratio"], bins=bins, labels=labels, right=False, include_lowest=True)

fig, ax = plt.subplots(figsize=(12, 10))
elderly_view.plot(column="elderly_group", categorical=True, cmap="YlOrRd", legend=True,
                  edgecolor="white", linewidth=.15, missing_kwds={"color": "lightgrey", "label": "Missing data"},
                  legend_kwds={"title": "Population aged 65+", "loc": "lower left"}, ax=ax)
ax.set_title(f"Share of Population Aged 65+\n{study['place_name']}, {study['year']}")
ax.set_axis_off(); plt.tight_layout(); plt.show()

C:\Users\25145857r\AppData\Local\Temp\ipykernel_12620\1491852468.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_axis_off(); plt.tight_layout(); plt.show()


### Landsat land-surface temperature

In [14]:
fig, ax = plt.subplots(figsize=(12, 10))
areas.plot(column="heat_exposure", cmap="inferno", legend=True, edgecolor="white", linewidth=.15,
           legend_kwds={"label": "Mean land-surface temperature (C)"}, ax=ax)
ax.set_title(f"Median Summer Land-Surface Temperature\n{study['place_name']}")
ax.set_axis_off(); plt.tight_layout(); plt.show()

C:\Users\25145857r\AppData\Local\Temp\ipykernel_12620\3083837866.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_axis_off(); plt.tight_layout(); plt.show()


In [15]:
heat_result = run_tool("summarize_heat_exposure", limit=10)
heat_summary = ctx.get("heat_summary")
print("heat artifacts:")
for name in ("map", "analysis_chart", "report"):
    print(" ", name, heat_summary[name])
print("spatial statistics:", heat_summary["statistics"])

summarize_heat_exposure ok=True {'top_tracts': [{'GEOID': '06037901216', 'heat_exposure': 59.13543660807518}, {'GEOID': '06037900901', 'heat_exposure': 58.3117344415703}, {'GEOID': '06037901214', 'heat_exposure': 58.28156518683863}, {'GEOID': '06037900902', 'heat_exposure': 58.16286518563603}, {'GEOID': '06037900102', 'heat_exposure': 58.0701043299373}, {'GEOID': '06037900301', 'heat_exposure': 57.94770931932072}, {'GEOID': '06037900103', 'heat_exposure': 57.89803685897436}, {'GEOID': '06037911100', 'heat_exposure': 57.358741010388}, {'GEOID': '06037901300', 'heat_exposure': 57.22794941571954}, {'GEOID': '06037900704', 'heat_exposure': 56.93461277173913}], 'tracts_reviewed': 2476, 'statistics': {'tracts': 2476, 'minimum_c': 33.77856167879972, 'maximum_c': 59.13543660807518, 'mean_c': 46.6737540645843, 'median_c': 46.97722308819111, 'standard_deviation_c': 3.3981372406180754, 'morans_i': 0.8494872582173877, 'permutation_p_value': 0.005, 'permutations': 199, 'islands': 0}, 'map': 'D:\\Re

## 5. Fetch candidate facilities

In [16]:
run_tool("fetch_candidate_sites", amenities=task.candidate_amenities)
candidates = ctx.get("candidates")
print("candidate count:", len(candidates))
print(candidates[[column for column in ("cand_id", "name", "amenity") if column in candidates]].head())

fetch_candidate_sites ok=True {'candidates': 4251, 'amenities': ['library', 'community_centre', 'school']}
candidate count: 4251
  cand_id                                       name           amenity
0       0                      YMCA of the Foothills  community_centre
1       1  Westchester-Loyola Village Branch Library           library
2       2                    Sunland-Tujunga Library           library
3       3                               Camp Hi-Hill            school
4       4               Beverly Hills Public Library           library


In [17]:
fig, ax = plt.subplots(figsize=(12, 10))
areas.boundary.plot(ax=ax, color="lightgrey", linewidth=.15)
candidates.to_crs(areas.crs).plot(ax=ax, color="#185FA5", markersize=8, alpha=.7, label="Candidate facilities")
ax.set_title(f"Candidate Cooling-Center Facilities\n{study['place_name']}")
ax.legend(); ax.set_axis_off(); plt.tight_layout(); plt.show()

C:\Users\25145857r\AppData\Local\Temp\ipykernel_12620\2136138842.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); ax.set_axis_off(); plt.tight_layout(); plt.show()


## 6. Build the travel matrix

The registered tool chooses Haversine or guarded sparse ORS from `cfg`. ORS requests are place-scoped and reuse cached pairs.

In [18]:
run_tool("build_travel_matrix")
travel = ctx.get("travel")
matrix = travel["matrix"]
print("router:", travel["router"])
print("shape:", matrix.shape)
print("diagnostics:", travel["diagnostics"])
assert matrix.shape == (len(areas), len(candidates))

build_travel_matrix ok=True {'router': 'haversine', 'shape': (2476, 4251), 'finite_pair_ratio': 1.0, 'unreachable_demand_count': 0, 'min_minutes': 0.030202148229063846, 'max_minutes': 2997.9787688508018, 'nearest_median_minutes': 6.608205059524786, 'nearest_p95_minutes': 22.49044911596625}
router: haversine
shape: (2476, 4251)
diagnostics: {'shape': (2476, 4251), 'finite_pair_ratio': 1.0, 'unreachable_demand_count': 0, 'min_minutes': 0.030202148229063846, 'max_minutes': 2997.9787688508018, 'nearest_median_minutes': 6.608205059524786, 'nearest_p95_minutes': 22.49044911596625}


In [19]:
for threshold in (10, 15, 20, 30):
    reachable = (matrix <= threshold).any(axis=1)
    ratio = population[reachable].sum() / population.sum()
    print(f"{threshold:2d} min: {ratio:.2%} population reachable")

10 min: 76.00% population reachable
15 min: 90.35% population reachable
20 min: 94.51% population reachable
30 min: 97.43% population reachable


## 7. Optimize and inspect the Pareto frontier

In [20]:
run_tool(
    "optimize_sites",
    n_facilities=task.n_facilities,
    threshold_minutes=task.threshold_minutes,
    weights=task.weights.model_dump(),
)
analysis = ctx.get("analysis")
solution = analysis["solution"]
frontier = analysis["frontier"]
print(solution.model_dump())
print("frontier points:", len(frontier))

optimize_sites ok=True {'selected_ids': ['148', '180', '1101', '1593', '1823', '2578', '2580', '2668', '3298', '4057'], 'covered_ratio': 0.03851020812765619, 'vulnerable_covered_ratio': 0.1250515421933303, 'covered_tract_ratio': 0.04402261712439418, 'all_tract_ratio': 0.04402261712439418, 'frontier_points': 6, 'requested_facilities': 10, 'candidate_count': 4251, 'coverage_target': None, 'coverage_group': 'all', 'max_tract_coverage_at_threshold': 0.8962035541195477, 'minimum_threshold_with_all_candidates': 376.8771800529564}
{'profile_name': 'requested', 'selected_ids': ['148', '180', '1101', '1593', '1823', '2578', '2580', '2668', '3298', '4057'], 'covered_population': 382664.0, 'covered_ratio': 0.03851020812765619, 'vulnerable_covered_ratio': 0.1250515421933303, 'mean_travel_minutes': 10.399855113495434, 'weights_used': {'population_density': 0.4, 'elderly_ratio': 0.2, 'poverty_rate': 0.2, 'heat_exposure': 0.2}, 'solver_status': 'Optimal'}
frontier points: 6


In [21]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter([point.efficiency_metric for point in analysis["points"]],
           [point.equity_metric for point in analysis["points"]], alpha=.35, label="Swept solutions")
ax.plot([point.efficiency_metric for point in frontier],
        [point.equity_metric for point in frontier], "o-", color="#EF9F27", label="Non-dominated frontier")
ax.set(xlabel="Total population coverage", ylabel="Vulnerable population coverage",
       title=f"Efficiency-Equity Frontier: N={task.n_facilities}, threshold={task.threshold_minutes:g} min")
ax.grid(alpha=.3); ax.legend(); plt.tight_layout(); plt.show()

C:\Users\25145857r\AppData\Local\Temp\ipykernel_12620\3015643907.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.grid(alpha=.3); ax.legend(); plt.tight_layout(); plt.show()


## 8. Generate production-format outputs

In [22]:
run_tool("generate_outputs")
outputs = ctx.get("outputs")
for name, output_path in outputs.items():
    print(name, output_path, "exists=" + str(Path(output_path).is_file()))
assert all(Path(output_path).is_file() for output_path in outputs.values())

generate_outputs ok=True {'map': 'D:\\Research\\ACM SIGSPATIAL\\oasis-agent\\outputs_run\\notebook\\los_angeles_county_california_usa\\coverage_map.html', 'pareto': 'D:\\Research\\ACM SIGSPATIAL\\oasis-agent\\outputs_run\\notebook\\los_angeles_county_california_usa\\pareto.png', 'report': 'D:\\Research\\ACM SIGSPATIAL\\oasis-agent\\outputs_run\\notebook\\los_angeles_county_california_usa\\report.md'}
map D:\Research\ACM SIGSPATIAL\oasis-agent\outputs_run\notebook\los_angeles_county_california_usa\coverage_map.html exists=True
pareto D:\Research\ACM SIGSPATIAL\oasis-agent\outputs_run\notebook\los_angeles_county_california_usa\pareto.png exists=True
report D:\Research\ACM SIGSPATIAL\oasis-agent\outputs_run\notebook\los_angeles_county_california_usa\report.md exists=True


The fixed CLI pipeline calls the same six registered tools in this order. Agent mode adds typed parsing, observation-driven tool selection, bounded correction, HITL, and evidence-grounded interpretation around these deterministic operations.